# Agreeableness Audit Pipeline

Audit LLM judge quality for the agreeableness eval (IPIP-NEO inspired, 6 facets, 36 items).

The agreeableness eval lives outside niels' `EvalConfig`, so this notebook loads data directly
from the notebook's CSV output rather than going through `bridge.py`.

Steps:
1. Setup & install deps
2. Load agreeableness results
3. Stratified sampling
4. Run alternative judges
5. Human annotation (optional)
6. Analysis & summary

## 1. Setup

In [ ]:
!pip install -q pyyaml pandas numpy scipy scikit-learn tenacity tqdm openai anthropic python-dotenv

In [ ]:
import os
from dotenv import load_dotenv
from google.colab import drive, userdata

drive.mount('/content/drive')

os.environ["OPENROUTER_API_KEY"] = userdata.get("openrouter")
os.environ["ANTHROPIC_API_KEY"] = userdata.get("anthropic")

load_dotenv()

In [ ]:
os.chdir('/content/drive/MyDrive/spar-ood-propensities/june/vibes_audit')

In [ ]:
from audit_config import from_yaml
from sample_for_review import load_data, stratified_sample
from run_alt_judges import run_judges
from analyze import inter_judge_correlations, bias_probes, audit_summary, gwets_ac2, cohen_weighted_kappa, score_to_bins, confusion_matrix_plot
from pathlib import Path
import pandas as pd
import numpy as np

CONFIG_PATH = Path("configs/agreeableness.yaml")
AGREEABLENESS_DIR = Path("../agreeableness")
OUTPUT_DIR = Path("output/agreeableness")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

config = from_yaml(CONFIG_PATH, output_dir=str(OUTPUT_DIR))
METRIC = config.score_column

print(f"Propensity: {config.display_name}")
print(f"Primary metric: {METRIC}")
print(f"Alt judges: {[j['name'] for j in config.alt_judges]}")

## 2. Load Agreeableness Results

Load the results CSV produced by `agreeableness_analysis.ipynb`. The notebook outputs `results_with_alignment.csv` (or `results.csv` without alignment/coherence columns).

In [ ]:
# Load results — prefer results_with_alignment.csv, fall back to results.csv
results_path = AGREEABLENESS_DIR / "results_with_alignment.csv"
if not results_path.exists():
    results_path = AGREEABLENESS_DIR / "results.csv"
    print(f"Using fallback: {results_path}")

results_df = pd.read_csv(results_path, low_memory=False)

# Rename answer -> response for audit framework compatibility
if "answer" in results_df.columns and "response" not in results_df.columns:
    results_df = results_df.rename(columns={"answer": "response"})

print(f"Loaded {len(results_df)} rows from {results_path.name}")
print(f"Groups: {results_df['group'].unique().tolist()}")
print(f"Facets: {results_df['facet_name'].unique().tolist()}")
print(f"\nScore stats ({METRIC}):")
print(results_df[METRIC].describe())

In [ ]:
# Score distributions by group
results_df.groupby('group')[METRIC].describe().round(1)

## 3. Stratified Sampling

In [ ]:
# Save consolidated results and run stratified sampling
data_path = OUTPUT_DIR / "all_results.csv"
results_df.to_csv(data_path, index=False)

config = from_yaml(CONFIG_PATH, data_path=str(data_path), output_dir=str(OUTPUT_DIR))
TARGET_N = 100
config.target_n = min(TARGET_N, len(results_df))

df_loaded = load_data(config)
sample = stratified_sample(df_loaded, config)

print(f"\nSampled: {len(sample)} rows")

# Save full sample
full_path = OUTPUT_DIR / f"sample_{len(sample)}.csv"
sample.to_csv(full_path, index=False)
print(f"Saved: {full_path}")

# Save blind sample (for human annotation)
blind_cols = ["question", "response"] + [
    c for c in config.metadata_columns if c in sample.columns
]
blind = sample[blind_cols].copy()
blind["human_label"] = ""
blind_path = OUTPUT_DIR / f"sample_{len(sample)}_blind.csv"
blind.to_csv(blind_path, index=False)
print(f"Saved blind: {blind_path}")

In [ ]:
# Preview the sample
sample[["question", "response", METRIC, "group", "facet_name"]].head()

## 4. Run Alternative Judges

Requires `OPENAI_API_KEY` and/or `ANTHROPIC_API_KEY` set above.

In [ ]:
sample_df = pd.read_csv(full_path, low_memory=False)
config = from_yaml(CONFIG_PATH, output_dir=str(OUTPUT_DIR))

print(f"Running alt judges on {len(sample_df)} rows...")
print(f"Judges: {[j['name'] for j in config.alt_judges]}")

result = run_judges(sample_df, config)

alt_path = OUTPUT_DIR / "alt_judge_scores.csv"
result.to_csv(alt_path, index=False)
print(f"\nSaved: {alt_path}")

In [ ]:
# Quick correlation check
score_cols = [c for c in result.columns if c.endswith("_score") and c != config.score_column]
for col in score_cols:
    valid = result[col].notna() & result[config.score_column].notna()
    if valid.sum() > 0:
        corr = result.loc[valid, col].corr(result.loc[valid, config.score_column])
        print(f"Correlation {config.score_column} vs {col}: {corr:.3f}")

## 5. Human Annotation (Optional)

The annotation GUI requires a local server. In Colab, you can review samples manually instead.

To use the full GUI locally:
```bash
cd june/vibes_audit
python annotate.py --config configs/agreeableness.yaml --output-dir output/agreeableness
```

In [ ]:
# Manual review: inspect a few samples
blind_df = pd.read_csv(blind_path)
for i, row in blind_df.head(5).iterrows():
    print(f"\n{'='*60}")
    print(f"Sample {i} | group={row.get('group', '?')} | facet={row.get('facet_name', '?')}")
    print(f"{'='*60}")
    print(f"Q: {str(row['question'])[:200]}...")
    print(f"\nA: {str(row['response'])[:300]}...")

### 5b. Load Human Annotations

After running the annotation GUI locally, upload or load `human_annotations.csv`.

In [ ]:
ann_path = OUTPUT_DIR / "human_annotations.csv"

# In Colab, upload the file if running remotely:
# from google.colab import files
# uploaded = files.upload()  # upload human_annotations.csv
# import shutil; shutil.move("human_annotations.csv", str(ann_path))

if not ann_path.exists():
    print(f"No annotations found at {ann_path}")
    print("Run the annotator locally first:")
    print(f"  python annotate.py --config configs/agreeableness.yaml --output-dir output/agreeableness")
else:
    human_df = pd.read_csv(ann_path)
    n_labeled = human_df["human_label"].notna() & (human_df["human_label"] != "")
    print(f"Loaded {n_labeled.sum()} / {len(human_df)} annotations from {ann_path}")
    print(f"\nLabel distribution:")
    print(human_df["human_label"].value_counts())

### Human vs Judge Agreement

Compare human labels against the original judge scores and alt judge scores.

In [ ]:
if ann_path.exists():
    import matplotlib.pyplot as plt
    from scipy import stats

    human_df = pd.read_csv(ann_path)
    config = from_yaml(CONFIG_PATH, output_dir=str(OUTPUT_DIR))

    # Map human labels to numeric (bucket number)
    label_to_num = {b.label: b.number for b in config.buckets}
    label_to_num[config.INCOHERENT_LABEL] = 0

    labeled = human_df[human_df["human_label"].notna() & (human_df["human_label"] != "")].copy()
    labeled["human_score"] = labeled["human_label"].map(label_to_num)
    labeled = labeled.dropna(subset=["human_score"])
    labeled["human_score"] = labeled["human_score"].astype(int)

    print(f"Analyzing {len(labeled)} labeled samples\n")

    # Merge with full sample to get judge scores
    sample_df = pd.read_csv(full_path, low_memory=False)
    if "index" in labeled.columns:
        labeled = labeled.set_index("index")
    merged = sample_df.loc[labeled.index].copy()
    merged["human_label"] = labeled["human_label"].values
    merged["human_score"] = labeled["human_score"].values

    # Filter out INCOHERENT for numeric comparisons
    valid = merged[merged["human_score"] > 0].copy()
    print(f"Valid for numeric comparison: {len(valid)} (excluded {len(merged) - len(valid)} INCOHERENT)\n")

    if len(valid) == 0:
        print("No valid samples for numeric comparison after filtering.")
    else:
        # Bin judge scores to same 5-bucket scale
        bucket_edges = [0, 20, 40, 60, 80, 100]
        bucket_labels = [1, 2, 3, 4, 5]
        valid["judge_bucket"] = pd.cut(
            valid[METRIC], bins=bucket_edges, labels=bucket_labels, include_lowest=True
        ).astype(int)

        print("=" * 60)
        print("HUMAN vs ORIGINAL JUDGE AGREEMENT")
        print("=" * 60)

        ac2 = gwets_ac2(valid["human_score"].tolist(), valid["judge_bucket"].tolist(), bucket_labels)
        print(f"  Gwet's AC2:             {ac2:.3f}")

        wk = cohen_weighted_kappa(valid["human_score"].values, valid["judge_bucket"].values, 5)
        print(f"  Weighted Cohen's Kappa: {wk:.3f}")

        exact = (valid["human_score"] == valid["judge_bucket"]).mean()
        print(f"  Exact match:            {exact:.1%}")

        within1 = (abs(valid["human_score"] - valid["judge_bucket"]) <= 1).mean()
        print(f"  Within 1 bucket:        {within1:.1%}")

        r, p = stats.spearmanr(valid["human_score"], valid[METRIC])
        print(f"  Spearman (human vs raw): r={r:.3f}, p={p:.4f}")

        # Confusion matrix
        bucket_short_labels = [config.buckets[len(config.buckets) - i].short for i in bucket_labels]

        fig, ax = plt.subplots(figsize=(7, 6))
        confusion_matrix_plot(
            valid["judge_bucket"].values,
            valid["human_score"].values,
            labels=bucket_labels,
            title=f"Agreeableness: Judge Bucket vs Human Label",
            ax=ax
        )
        ax.set_xlabel("Human Label (bucket)")
        ax.set_ylabel("Judge Score (binned)")
        ax.set_xticks(bucket_labels)
        ax.set_xticklabels(bucket_short_labels)
        ax.set_yticks(bucket_labels)
        ax.set_yticklabels(bucket_short_labels)
        plt.tight_layout()
        plt.show()
else:
    print("Skipping — no human annotations found.")

In [ ]:
# Disagreement analysis: which samples did human and judge disagree on most?
if ann_path.exists() and 'valid' in dir() and len(valid) > 0:
    valid["disagreement"] = abs(valid["human_score"] - valid["judge_bucket"])
    disagreed = valid[valid["disagreement"] >= 2].sort_values("disagreement", ascending=False)

    print(f"Large disagreements (>= 2 buckets apart): {len(disagreed)} / {len(valid)}")
    print()

    for i, (_, row) in enumerate(disagreed.head(10).iterrows()):
        print(f"--- Disagreement #{i+1}: human={int(row['human_score'])} vs judge_bucket={int(row['judge_bucket'])} (raw={row[METRIC]:.0f}) ---")
        print(f"  Group: {row.get('group', '?')} | Facet: {row.get('facet_name', '?')}")
        print(f"  Q: {str(row['question'])[:150]}...")
        print(f"  A: {str(row['response'])[:200]}...")
        print()

### Systematic Bias Analysis

Quantify the direction and source of disagreements between human and judge.

In [ ]:
if ann_path.exists() and 'valid' in dir() and len(valid) > 0:
    import matplotlib.pyplot as plt
    from scipy import stats

    # --- Directional bias ---
    valid["delta"] = valid["human_score"] - valid["judge_bucket"]  # positive = judge too low
    mean_delta = valid["delta"].mean()
    judge_too_low = (valid["delta"] > 0).sum()
    judge_too_high = (valid["delta"] < 0).sum()
    exact_agree = (valid["delta"] == 0).sum()

    print("=" * 60)
    print("DIRECTIONAL BIAS (human - judge_bucket)")
    print("=" * 60)
    print(f"  Mean delta:       {mean_delta:+.2f} buckets {'(judge scores too LOW)' if mean_delta > 0 else '(judge scores too HIGH)'}")
    print(f"  Judge too low:    {judge_too_low} ({judge_too_low/len(valid):.0%})")
    print(f"  Exact agreement:  {exact_agree} ({exact_agree/len(valid):.0%})")
    print(f"  Judge too high:   {judge_too_high} ({judge_too_high/len(valid):.0%})")

    # Wilcoxon signed-rank test
    if len(valid[valid["delta"] != 0]) >= 10:
        stat, p = stats.wilcoxon(valid["delta"])
        print(f"\n  Wilcoxon signed-rank: W={stat:.0f}, p={p:.4f}")
        print(f"  {'Significant systematic bias' if p < 0.05 else 'No significant systematic bias'} (alpha=0.05)")

    # --- Distribution of deltas ---
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Histogram of deltas
    ax = axes[0]
    delta_counts = valid["delta"].value_counts().sort_index()
    colors = ["#da3633" if d < 0 else "#238636" if d > 0 else "#8b949e" for d in delta_counts.index]
    ax.bar(delta_counts.index, delta_counts.values, color=colors, edgecolor="white", linewidth=0.5)
    ax.axvline(0, color="white", linewidth=1, linestyle="--", alpha=0.5)
    ax.set_xlabel("Human - Judge (buckets)")
    ax.set_ylabel("Count")
    ax.set_title("Disagreement Direction\n(positive = judge too low, negative = judge too high)")
    ax.set_xticks(range(int(valid["delta"].min()), int(valid["delta"].max()) + 1))

    # Scatter: raw judge score vs human bucket
    ax = axes[1]
    jitter = np.random.RandomState(42).uniform(-0.15, 0.15, len(valid))
    ax.scatter(valid["human_score"] + jitter, valid[METRIC], alpha=0.5, s=30, c="#58a6ff", edgecolors="none")
    for edge in bucket_edges[1:-1]:
        ax.axhline(edge, color="#30363d", linewidth=0.8, linestyle="--")
    ax.set_xlabel("Human Label (bucket)")
    ax.set_ylabel(f"Judge Raw Score ({METRIC})")
    ax.set_title("Human Label vs Raw Judge Score")
    ax.set_xticks([1, 2, 3, 4, 5])
    ax.set_xticklabels(["Very\nDisag", "Disag", "Bal", "Agree", "Very\nAgree"], fontsize=9)

    plt.tight_layout()
    plt.show()

In [ ]:
if ann_path.exists() and 'valid' in dir() and len(valid) > 0:
    # --- Breakdown by group ---
    print("=" * 60)
    print("DISAGREEMENT BY GROUP")
    print("=" * 60)

    group_stats = valid.groupby("group").agg(
        n=("delta", "size"),
        mean_delta=("delta", "mean"),
        large_disagree=("disagreement", lambda x: (x >= 2).sum()),
        mean_judge=(METRIC, "mean"),
        mean_human=("human_score", "mean"),
    ).sort_values("mean_delta", ascending=False)
    group_stats["large_disagree_pct"] = (group_stats["large_disagree"] / group_stats["n"] * 100).round(1)
    group_stats["mean_delta"] = group_stats["mean_delta"].round(2)
    group_stats["mean_judge"] = group_stats["mean_judge"].round(1)
    group_stats["mean_human"] = group_stats["mean_human"].round(2)

    print(group_stats.to_string())

    # --- Breakdown by facet ---
    if "facet_name" in valid.columns:
        print(f"\n{'=' * 60}")
        print("DISAGREEMENT BY FACET")
        print("=" * 60)

        facet_stats = valid.groupby("facet_name").agg(
            n=("delta", "size"),
            mean_delta=("delta", "mean"),
            large_disagree=("disagreement", lambda x: (x >= 2).sum()),
            mean_judge=(METRIC, "mean"),
            mean_human=("human_score", "mean"),
        ).sort_values("mean_delta", ascending=False)
        facet_stats["large_disagree_pct"] = (facet_stats["large_disagree"] / facet_stats["n"] * 100).round(1)
        facet_stats["mean_delta"] = facet_stats["mean_delta"].round(2)
        facet_stats["mean_judge"] = facet_stats["mean_judge"].round(1)
        facet_stats["mean_human"] = facet_stats["mean_human"].round(2)

        print(facet_stats.to_string())

    # --- Plot ---
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # By group
    ax = axes[0]
    gs = group_stats.sort_values("mean_delta")
    colors = ["#da3633" if d < -0.3 else "#238636" if d > 0.3 else "#9e6a03" for d in gs["mean_delta"]]
    ax.barh(range(len(gs)), gs["mean_delta"], color=colors)
    ax.set_yticks(range(len(gs)))
    ax.set_yticklabels(gs.index, fontsize=9)
    ax.axvline(0, color="white", linewidth=0.8)
    ax.set_xlabel("Mean Delta (human - judge bucket)")
    ax.set_title("Judge Bias by Group\n(positive = judge too low)")

    # By facet
    if "facet_name" in valid.columns:
        ax = axes[1]
        fs = facet_stats.sort_values("mean_delta")
        colors = ["#da3633" if d < -0.3 else "#238636" if d > 0.3 else "#9e6a03" for d in fs["mean_delta"]]
        ax.barh(range(len(fs)), fs["mean_delta"], color=colors)
        ax.set_yticks(range(len(fs)))
        ax.set_yticklabels([f.title() for f in fs.index], fontsize=9)
        ax.axvline(0, color="white", linewidth=0.8)
        ax.set_xlabel("Mean Delta (human - judge bucket)")
        ax.set_title("Judge Bias by Facet\n(positive = judge too low)")

    plt.tight_layout()
    plt.show()

## 6. Analysis & Summary

In [ ]:
alt_df = pd.read_csv(alt_path, low_memory=False)
config = from_yaml(CONFIG_PATH, output_dir=str(OUTPUT_DIR))

# Check for human annotations
ann_path = OUTPUT_DIR / "human_annotations.csv"
human_df = pd.read_csv(ann_path) if ann_path.exists() else pd.DataFrame()

# Audit summary
summary = audit_summary(config, human_df, alt_df)
summary["eval"] = "agreeableness"

summary_path = OUTPUT_DIR / "audit_summary.csv"
summary.to_csv(summary_path, index=False)

print(f"Agreeableness Audit Summary:")
print(f"{'='*70}")
for _, row in summary.iterrows():
    icon = {"PASS": "PASS", "MARGINAL": "WARN", "FAIL": "FAIL"}.get(row["Status"], "?")
    print(f"  [{icon:>4}] {row['Metric']}: {row['Value']} (threshold {row['Threshold']})")

summary

In [ ]:
# Bias probes
group_cols = [c for c in config.metadata_columns if c in alt_df.columns]
probes = bias_probes(alt_df, config.score_column, group_cols)

print("Bias Probes:")
print(f"{'='*70}")
for probe_name, res in probes.items():
    if "r" in res:
        print(f"  {probe_name}: r={res['r']:.3f}, p={res['p']:.4f}")
    elif "F" in res:
        print(f"  {probe_name}: F={res['F']:.2f}, p={res['p']:.4f}")
        if "group_means" in res:
            for gname, gmean in res["group_means"].items():
                print(f"    {gname}: mean={gmean:.1f}")

In [ ]:
# Inter-judge correlations
all_score_cols = [config.score_column] + [
    c for c in alt_df.columns if c.endswith("_score") and c != config.score_column
]
corr_df = inter_judge_correlations(alt_df, all_score_cols)
print("Inter-Judge Correlations:")
corr_df